# CÂU 7: PHÂN TÍCH TỔNG DOANH THU THEO VÙNG ĐỊA LÝ

**1. Yêu cầu bài toán:**
Kết nối dữ liệu để xác định vùng địa lý (`region`) nào mang lại tổng doanh thu cao nhất.

**2. Phương pháp tiếp cận & Suy luận:**
Doanh thu không nằm sẵn cùng với Vùng địa lý, nên ta cần thực hiện quy trình sau:
* **Bước 1:** Tải dữ liệu từ 3 file: `geography.csv`, `orders.csv`, và `order_items.csv`.
* **Bước 2:** Tính doanh thu cho từng dòng sản phẩm trong đơn hàng. Công thức: `Doanh thu = (Số lượng * Đơn giá) - Giảm giá`.
* **Bước 3:** Nối (Join) bảng `order_items` với `orders` (thông qua `order_id`) để lấy mã bưu chính (`zip`) của từng đơn.
* **Bước 4:** Nối tiếp bảng kết quả với `geography` (thông qua `zip`) để lấy tên vùng (`region`).
* **Bước 5:** Gom nhóm dữ liệu theo `region`, tính tổng (`sum`) doanh thu và sắp xếp để tìm vùng cao nhất.

**3. Lý do chọn phương pháp:**
* **Sử dụng `order_items` thay vì `payments`:** Bảng `payments` chứa tổng tiền khách trả nhưng có thể bao gồm cả phí vận chuyển (shipping fee). Việc tính doanh thu thuần trực tiếp từ `order_items` sẽ phản ánh chính xác nhất giá trị thực của hàng hóa bán ra.
* **Xử lý giá trị khuyết (NaN) bằng `fillna(0)`:** Cột `discount_amount` có thể bị trống nếu khách không dùng mã giảm giá. Việc chuyển đổi các ô trống này thành số 0 là phép toán an toàn bắt buộc trước khi làm phép trừ, tránh lỗi toàn bộ doanh thu bị biến thành NaN.
* **Sử dụng `pd.merge()` chuỗi:** Kỹ thuật join lần lượt từng cặp bảng giúp kiểm soát dữ liệu chặt chẽ, hoạt động giống như lệnh `JOIN` với nhiều mệnh đề `ON` trong SQL.

In [1]:
# Import thư viện
import pandas as pd
from google.colab import drive

# Kết nối với Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


**Thực hiện Bước 1 & 2: Tải dữ liệu và Tính toán Doanh thu từng dòng**

Chúng ta đọc 3 bảng dữ liệu cần thiết, xử lý các ô giảm giá bị trống và thiết lập cột 'revenue' (doanh thu) cho từng sản phẩm.

In [2]:
# 1. Khai báo đường dẫn
path_geo = '/content/drive/MyDrive/datathon-2026-round-1/geography.csv'
path_orders = '/content/drive/MyDrive/datathon-2026-round-1/orders.csv'
path_items = '/content/drive/MyDrive/datathon-2026-round-1/order_items.csv'

# 2. Đọc dữ liệu
df_geo = pd.read_csv(path_geo)
df_orders = pd.read_csv(path_orders)
df_items = pd.read_csv(path_items)

# 3. Tính doanh thu cho từng dòng (Xử lý NaN ở cột discount_amount thành 0)
df_items['discount_amount'] = df_items['discount_amount'].fillna(0)
df_items['revenue'] = (df_items['quantity'] * df_items['unit_price']) - df_items['discount_amount']

# Gom nhóm theo đơn hàng để tính tổng doanh thu của cả đơn đó
order_revenue = df_items.groupby('order_id')['revenue'].sum().reset_index()

/tmp/ipykernel_5218/4081233907.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(path_items)


**Thực hiện Bước 3, 4 & 5: Nối bảng đa tầng và Thống kê tổng**

Sử dụng `merge` để gắn mã zip vào đơn hàng, sau đó gắn tên vùng vào mã zip. Cuối cùng, tổng hợp doanh thu.

In [3]:
# Nối bảng Doanh thu đơn hàng với bảng Đơn hàng (để lấy cột zip)
df_order_zip = pd.merge(order_revenue, df_orders[['order_id', 'zip']], on='order_id', how='inner')

# Nối bảng vừa xong với bảng Địa lý (để lấy cột region)
df_final = pd.merge(df_order_zip, df_geo[['zip', 'region']], on='zip', how='inner')

# Gom nhóm theo vùng và tính tổng doanh thu
region_revenue = df_final.groupby('region')['revenue'].sum().sort_values(ascending=False)

print("Tổng doanh thu theo từng vùng địa lý:")
print(region_revenue)

# Xác định vùng cao nhất
best_region = region_revenue.idxmax()
max_revenue = region_revenue.max()

print(f"\n--> Vùng mang lại tổng doanh thu cao nhất là: '{best_region}' (với {max_revenue:,.2f})")

Tổng doanh thu theo từng vùng địa lý:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: revenue, dtype: float64

--> Vùng mang lại tổng doanh thu cao nhất là: 'East' (với 7,291,150,819.12)


**KẾT LUẬN CUỐI CÙNG:**
Dựa trên kết quả tính toán sau khi kết nối ba bảng dữ liệu, vùng mang lại tổng doanh thu cao nhất là vùng **East** (với doanh thu xấp xỉ 7,291,150,819).

Đối chiếu với các phương án của đề bài:

A) West

B) Central

C) East

D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

**=> Đáp án được chọn là: C**